# Augmenting the BEMO Prompts Dataset

### Loading the BEMO Prompts Dataset & Understanding the Data

In [1]:
import pandas as pd

df = pd.read_csv('bemo_prompts_dataset_v2.csv')

In [2]:
df.sample(5)

,prompt,general,smart_home,todo,mail,others
158,"Start the vacuum cleaner, add 'finish the proj...",0,1,1,1,0
251,Add 'finish the project' to my tasks.,0,0,1,0,0
231,Open the garage door and set a task to renew m...,0,1,1,0,0
5,Add 'buy groceries' to my to-do list.,0,0,1,0,0
111,What is your main function?,0,0,0,0,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   prompt      300 non-null    object
 1   general     300 non-null    int64 
 2   smart_home  300 non-null    int64 
 3   todo        300 non-null    int64 
 4   mail        300 non-null    int64 
 5   others      300 non-null    int64 
dtypes: int64(5), object(1)
memory usage: 14.2+ KB


In [4]:
# Get the sum of numerical columns
general_sum = df['general'].sum()
smart_home_sum = df['smart_home'].sum()
todo_sum = df['todo'].sum()
mail_sum = df['mail'].sum()
other_sum = df['others'].sum()

print(f"General: {general_sum}")
print(f"Smart Home: {smart_home_sum}")
print(f"Todo: {todo_sum}")
print(f"Mail: {mail_sum}")
print(f"Other: {other_sum}")

General: 131
Smart Home: 131
Todo: 131
Mail: 131
Other: 51


In [5]:
# Shuffle the data
df = df.sample(frac=1).reset_index(drop=True)

In [7]:
prompts = df['prompt'].values.tolist()
prompts[:5]

['How can I save money and set the mood lighting in the dining room.',
 'What are the benefits of meditation, turn off the kitchen lights, set a reminder to exercise at 6 PM, and send an invitation email for the event.',
 'Add a task to organize my files.',
 'Can you tell me a fun fact?',
 'How do I get started with programming, remind me to update my calendar, and send an email confirming the appointment.']

In [8]:
labels = df.drop(columns=["prompt"]).values.tolist()
labels[:5]

[[1, 1, 0, 0, 0],
 [1, 1, 1, 1, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 1, 1, 0]]

### Paraphrasing using gemini-1.5-flash

In [9]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values

load_dotenv()
os.environ["HUGGINGFACEHUB_API_TOKEN"] = dotenv_values(find_dotenv())[
    "HUGGINGFACE_API_KEY_TEST"
]
gemini_api_key = dotenv_values(find_dotenv())["GEMINI_API_KEY_TEST"]
os.environ["GOOGLE_API_KEY"] = gemini_api_key

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

system_prompt =  ChatPromptTemplate(
            [
                (
                    "system",
                    """
You are an advanced paraphrasing model. Your task is to generate coherent, meaningful paraphrased versions of the given sentence. Ensure the paraphrased sentences retain the original intent, meaning, and context of the input while matching the provided labels.

### Instructions:
1. Paraphrase the sentence into as many variations as you find sufficient to cover diverse phrasings.
2. Keep all paraphrases clear, grammatically correct, and contextually relevant.
3. The paraphrased sentences must still be suitable for the original label(s).

### Format:
Return the output as a JSON object with this structure:
{{"result": ["sentence 1", "sentence 2", "sentence 3", ...]}}

### Example Input:
Sentence: "Remind me to call John and send an email to Ahmed."
Labels: ["todo", "mail"]

### Example Output:
{{
  "result": [
    "Set a reminder for me to call John and email Ahmed.",
    "Don't forget to remind me to call John and shoot an email to Ahmed.",
    "Please remind me about my call with John and to send Ahmed an email.",
    "Help me remember to phone John and draft an email to Ahmed."
  ]
}}

### Now, paraphrase the following:
""",
),
("human", "{user_input}"),
("placeholder", "{messages}"),
            ]
        )

In [12]:
chain = system_prompt | llm | JsonOutputParser()

In [16]:
def gemini_paraphrase(sentence, labels):
    input_dict = {"sentence": sentence, "labels": labels}
    output = chain.invoke(input={"user_input": str(input_dict)})
    return output    

In [17]:
gemini_paraphrase("What is the capital of France?", ["general"])

{'result': ['Which city serves as the capital of France?',
  'Name the capital city of France.',
  'Can you tell me the capital of France?',
  'What city is the capital of France?',
  "France's capital city is which one?",
  'In France, what is the capital?',
  'Could you tell me the name of the French capital?']}

In [28]:
#! RUN THIS ONLY ONCE
last_i = 0
gemini_paraphrased_prompts = []
gemini_paraphrased_labels = []

In [171]:
for i in range(last_i, len(prompts)):
    print(f"Processing prompt {i+1}/{len(prompts)}")
    try:
        results = gemini_paraphrase(prompts[i], labels[i])["result"]
    except Exception as e:
        print(f"Error: {e}, Stopping at prompt {i}")
        last_i = i
        break
    gemini_paraphrased_prompts.extend(results)
    for _ in range(len(results)):
        gemini_paraphrased_labels.append(labels[i])

Processing prompt 300/300


In [179]:
if i == len(prompts) - 1:
    print("All prompts processed successfully")
    last_i = i + 1

All prompts processed successfully


In [180]:
print(len(gemini_paraphrased_prompts), len(gemini_paraphrased_labels))

2268 2268


In [181]:
prompts[:5]

['How can I save money and set the mood lighting in the dining room.',
 'What are the benefits of meditation, turn off the kitchen lights, set a reminder to exercise at 6 PM, and send an invitation email for the event.',
 'Add a task to organize my files.',
 'Can you tell me a fun fact?',
 'How do I get started with programming, remind me to update my calendar, and send an email confirming the appointment.']

In [182]:
labels[:5]

[[1, 1, 0, 0, 0],
 [1, 1, 1, 1, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 1, 1, 0]]

In [183]:
gemini_paraphrased_prompts[:5]

["What's the most cost-effective way to create the right ambiance with lighting in my dining room?",
 'I want to set a mood with lighting in my dining room, but I need to be budget-conscious. How can I do that?',
 'How can I achieve a pleasant dining room atmosphere with lighting while minimizing expenses?',
 'Affordable mood lighting for the dining room: any ideas?',
 "I'm looking for budget-friendly ways to enhance the atmosphere in my dining room using lighting.  Any suggestions?"]

In [184]:
gemini_paraphrased_labels[:5]

[[1, 1, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 0, 0, 0]]

In [185]:
print(last_i)
print(len(prompts) - last_i)

300
0


In [186]:
# Create a new dataframe
df_gemini = pd.DataFrame(
    {
        "prompt": gemini_paraphrased_prompts,
        "general": [label[0] for label in gemini_paraphrased_labels],
        "smart_home": [label[1] for label in gemini_paraphrased_labels],
        "todo": [label[2] for label in gemini_paraphrased_labels],
        "mail": [label[3] for label in gemini_paraphrased_labels],
        "others": [label[4] for label in gemini_paraphrased_labels],
    }
)

In [187]:
df_gemini.sample(5)

,prompt,general,smart_home,todo,mail,others
2234,I need to adjust the AC and call my mom tomorr...,0,1,1,0,0
562,I need to start vacuuming and add 'finishing t...,0,1,1,0,0
1313,"Please help me begin my programming journey, i...",1,1,1,0,0
2206,What are meditation's advantages? Please also...,1,1,0,1,0
1185,Are you capable of interpreting sarcasm?,0,0,0,0,1


In [189]:
df_gemini.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2268 entries, 0 to 2267
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   prompt      2268 non-null   object
 1   general     2268 non-null   int64 
 2   smart_home  2268 non-null   int64 
 3   todo        2268 non-null   int64 
 4   mail        2268 non-null   int64 
 5   others      2268 non-null   int64 
dtypes: int64(5), object(1)
memory usage: 106.4+ KB


In [190]:
df_gemini.describe()

,general,smart_home,todo,mail,others
count,2268.000000,2268.000000,2268.000000,2268.000000,2268.000000
mean,0.425926,0.432099,0.438713,0.421517,0.192681
std,0.494592,0.495477,0.496339,0.493911,0.394491
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000


In [191]:
# Get the sum of numerical columns
general_sum = df_gemini['general'].sum()
smart_home_sum = df_gemini['smart_home'].sum()
todo_sum = df_gemini['todo'].sum()
mail_sum = df_gemini['mail'].sum()
other_sum = df_gemini['others'].sum()

print(f"General: {general_sum}")
print(f"Smart Home: {smart_home_sum}")
print(f"Todo: {todo_sum}")
print(f"Mail: {mail_sum}")
print(f"Other: {other_sum}")

General: 966
Smart Home: 980
Todo: 995
Mail: 956
Other: 437


In [188]:
# Save the dataframe to a CSV file
df_gemini.to_csv("bemo_prompts_dataset_v2_gemini.csv", index=False)

In [198]:
# Load the dataframes from the CSV file
df_gemini = pd.read_csv("bemo_prompts_dataset_v2_gemini.csv")
df_og = pd.read_csv("bemo_prompts_dataset_v2.csv")

In [200]:
# Concatenate the dataframes
df_concat = pd.concat([df_og, df_gemini])

In [201]:
df_concat.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2568 entries, 0 to 2267
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   prompt      2568 non-null   object
 1   general     2568 non-null   int64 
 2   smart_home  2568 non-null   int64 
 3   todo        2568 non-null   int64 
 4   mail        2568 non-null   int64 
 5   others      2568 non-null   int64 
dtypes: int64(5), object(1)
memory usage: 140.4+ KB


In [202]:
df_concat.describe()

,general,smart_home,todo,mail,others
count,2568.000000,2568.000000,2568.000000,2568.000000,2568.000000
mean,0.427181,0.432632,0.438474,0.423287,0.190031
std,0.494765,0.495537,0.496297,0.494176,0.392402
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000


In [203]:
# Save the concatenated dataframe to a CSV file
df_concat.to_csv("bemo_prompts_dataset_v2_concat.csv", index=False)